# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All operations refer to dataset elements by their Croissant schema `@id`, ensuring reproducibility and compliance with FAIR principles.

### Dataset Source

Croissant schema: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Ensure `mlcroissant` is installed in your environment
!pip install -q mlcroissant

## 1. Data Loading
Load dataset metadata and records from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import numpy as np

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Fetch and pretty print metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Explore available record sets, their `@id`s, fields, and columns.

In [ ]:
# List all record sets available in the dataset, showing their @id and field @ids
record_sets = list(dataset.record_sets.values())

if not record_sets:
    print("No record sets found in the dataset. Attempting fallback via internal Dataset metadata.")
    # Sometimes the record_set property is not populated; let's print available keys and some metadata
    for key in dir(metadata):
        if key.startswith("record_set") or key.startswith("recordSet"):
            print(key, getattr(metadata, key))
else:
    for rs in record_sets:
        print(f"RecordSet '@id': {rs.id}")
        print(f"  Name: {rs.name}")
        print("  Fields:")
        for field in rs.fields:
            f_name = getattr(field, 'name', '<unnamed>')
            print(f"    - {field.id} (name: {f_name})")
        print()

Below, we will enumerate a preview of the first few records for each record set using their `@id`. All entity access is performed only by `@id` as required.

In [ ]:
# Preview sample records for each record set by @id
for rs_id in dataset.record_sets:
    print(f"--- Records from record set '@id': {rs_id} ---")
    records_iter = dataset.records(record_set=rs_id)
    for i, record in enumerate(records_iter):
        print(record)
        if i >= 2:
            print("(Only showing first 3 records)")
            break
    print()

## 3. Data Extraction

Let us extract each record set by its `@id` and directly build a pandas DataFrame for analysis.

We'll gather the available record sets' `@id`s and extract all records from each, referring to columns and fields by their exact Croissant `@id`.

In [ ]:
# Gather IDs of all record sets
record_set_ids = list(dataset.record_sets.keys())
print("Record set @ids found in dataset:")
print(record_set_ids)

# Extract each set into a dataframe
dataframes = {}
for rs_id in record_set_ids:
    print(f"\nLoading records from: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"  Columns: {list(df.columns)} | Rows: {len(df)}")

# For demonstration, show columns and head of the first record set (if exists)
if record_set_ids:
    example_rs = record_set_ids[0]
    print(f"\nExample record set: {example_rs}")
    print(dataframes[example_rs].columns.tolist())
    display(dataframes[example_rs].head())

## 4. Exploratory Data Analysis (EDA)

### Example: Filtering, Normalizing, and Grouping on Numeric and Categorical Fields

We choose a numeric field and a grouping field from the first record set, always referring to fields by their Croissant `@id`s.

In [ ]:
# Choose the main record set for analysis (first one by default)
record_set_id = record_set_ids[0] if record_set_ids else None
df = dataframes[record_set_id]

# Let's inspect field @ids available
print(f"Fields (@id) in selected RecordSet ({record_set_id}):")
for col in df.columns:
    print(f"- {col}")

# For demonstration, pick likely numeric and grouping fields by inspecting @ids
# (You may want to tailor the below lines once you know the concrete @ids)
# Replace the below with the appropriate @ids from your dataset.
"""
Suppose from the previous display you see these columns: 
['age@cr', 'sex@cr', 'msi_status@cr', 'anatomical_location@cr', ...]
Let's assume 'age@cr' stores age (numeric), and 'anatomical_location@cr' is a categorical variable.
"""
# These @ids are placeholders. Replace them with what you see in your output!
numeric_field_id = None
group_field_id = None

# Find a numeric field candidate
for col in df.columns:
    if 'age' in col.lower() or 'interval' in col.lower() or 'number' in col.lower():
        # Try to ensure it's numeric
        if np.issubdtype(df[col].dropna().astype('str').str.replace(',','').astype(float, errors='ignore').dtype, np.number):
            numeric_field_id = col
            break

# Find a group field candidate
for col in df.columns:
    if (('sex' in col.lower() or 'group' in col.lower() or 'msi_status' in col.lower())
        and col != numeric_field_id):
        group_field_id = col
        break

if not numeric_field_id:
    raise ValueError("Could not automatically determine a numeric field; please update the 'numeric_field_id' variable accordingly.")
if not group_field_id:
    raise ValueError("Could not automatically determine a group field; please update the 'group_field_id' variable accordingly.")

print(f"Using numeric field: {numeric_field_id}")
print(f"Using group field: {group_field_id}")

# Ensure numeric
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Remove outliers for demonstration (z-score threshold)
mean = df[numeric_field_id].mean()
std = df[numeric_field_id].std()
zscore = (df[numeric_field_id] - mean) / std
filtered_df = df[(zscore > -3) & (zscore < 3)]

# Normalization
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std

print(f"Filtered records (|z|<3) for {numeric_field_id}, normalized:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized", group_field_id]].head())

# Grouped analysis
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped mean '{numeric_field_id}' by '{group_field_id}':")
    display(grouped_df)

## 5. Visualization

Let's visualize the distribution of the selected numeric field—e.g., age distribution, with grouping by the selected category.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and group_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(data=filtered_df, x=numeric_field_id, hue=group_field_id, bins=15, kde=True, multiple="stack")
    plt.title(f"Distribution of {numeric_field_id} by {group_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(7, 4))
    sns.boxplot(data=filtered_df, x=group_field_id, y=numeric_field_id)
    plt.title(f"Boxplot of {numeric_field_id} grouped by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.tight_layout()
    plt.show()
else:
    print("Numeric and/or group field not properly set for visualization.")

## 6. Conclusion

In this notebook, we've demonstrated how to load and process a clinical, biomarker-rich dataset using the `mlcroissant` library with strict reference to entity `@id`s as defined by the Croissant schema. We:

- Listed available record sets and fields by `@id`.
- Loaded records directly from Croissant via `mlcroissant`.
- Performed basic EDA with normalization and grouping, strictly referencing variables by `@id`.
- Visualized data distributions and inter-group differences in the selected numeric feature.

For further exploration, you can tailor the analysis to clinical hypotheses—such as associations of MSI status with anatomical site, or distribution of ages by diagnosis, using the Croissant `@id`s as reliable references throughout.

For more on Croissant, see [the specification](https://mlcommons.github.io/croissant/).

---